# Baseline SleepFM sur PhysioNet 2018 — Google Colab

Premier vrai chiffre de stadification (5 classes : Wake, N1, N2, N3, REM) avec le
checkpoint officiel SleepFM, sur de vrais enregistrements PhysioNet 2018.

**Avant de lancer** : `Exécution > Modifier le type d'exécution > GPU T4`, puis
`Exécution > Tout exécuter`. Compter ~30 à 45 min pour 20 sujets (l'extraction
CPU est l'étape la plus longue, les embeddings sur GPU vont vite).

PhysioNet 2018 est en accès libre (pas de DUA), donc pas de restriction à
l'utiliser sur Colab — contrairement à MESA/SHHS/CHAT (NSRR).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "Pas de GPU : Exécution > Modifier le type d'exécution > GPU T4"
!df -h /content | tail -1

## 1. Code et dépendances

In [ ]:
%cd /content
!git clone -q https://github.com/sumarc183-design/sleepfm-pipeline.git || (cd sleepfm-pipeline && git pull -q)
%cd /content/sleepfm-pipeline
!pip install -q -r requirements.txt

## 2. Checkpoint officiel SleepFM (`best.pt`, ~84 Mo, versionné dans le dépôt officiel)

In [ ]:
![ -d third_party/sleepfm ] || git clone -q --depth 1 https://github.com/rthapa84/sleepfm-codebase.git third_party/sleepfm
!ls -lh third_party/sleepfm/sleepfm/checkpoint/best.pt

## 3. Téléchargement des sujets PhysioNet 2018

3 fichiers par sujet (`.hea`, `.mat`, `-arousal.mat`), ~130-160 Mo chacun.
Le jeu complet fait 994 sujets / ~135 Go : augmenter `N_SUBJECTS` progressivement
(le disque Colab gratuit tient sans problème 50 sujets ; le pipeline stocke les
époques extraites en float64, ~0,7 Go par sujet).

In [ ]:
N_SUBJECTS = 3

import os, subprocess, urllib.request
from concurrent.futures import ThreadPoolExecutor

BASE = "https://physionet.org/files/challenge-2018/1.0.0/training"
OUT = "data/raw/physionet2018/training"
ids = [l.strip().strip("/") for l in urllib.request.urlopen(f"{BASE}/RECORDS").read().decode().split() if l.strip()][:N_SUBJECTS]

def fetch(subject):
    os.makedirs(f"{OUT}/{subject}", exist_ok=True)
    for f in (f"{subject}.hea", f"{subject}.mat", f"{subject}-arousal.mat"):
        dest = f"{OUT}/{subject}/{f}"
        if not os.path.exists(dest) or os.path.getsize(dest) == 0:
            subprocess.run(["curl", "-sfL", "--retry", "3", "-o", dest, f"{BASE}/{subject}/{f}"], check=True)
    return subject

with ThreadPoolExecutor(8) as pool:
    for i, s in enumerate(pool.map(fetch, ids), 1):
        print(f"[{i}/{len(ids)}] {s}")
!du -sh {OUT}

## 4. Baseline : extraction → embeddings (GPU) → sonde logistique → évaluation

Split **par sujet** (jamais par époque) : 25 % des sujets en test.

In [ ]:
!python run_baseline.py \
    --raw-data data/raw/physionet2018/training \
    --checkpoint third_party/sleepfm/sleepfm/checkpoint/best.pt \
    --num-threads 2 \
    --batch-size 128 2>&1 | grep -v -i warning | tee results_physionet2018.txt

## 5. Garder le résultat

Colab efface tout à la fin de la session : télécharger `results_physionet2018.txt`
(panneau Fichiers à gauche) ou le copier sur Drive.

**Lecture du chiffre** : la sonde est une régression logistique sur embeddings
figés, comme le protocole officiel. Le papier SleepFM rapporte ses chiffres de
stadification sur MESA/SHHS, pas sur PhysioNet 2018 : ce kappa est une première
référence interne, la comparaison stricte au chiffre publié se fera sur MESA.

In [ ]:
# from google.colab import drive; drive.mount('/content/drive')
# !cp results_physionet2018.txt /content/drive/MyDrive/